# Tworzenie aplikacji do generowania obrazów (Azure OpenAI)

LLM-y to nie tylko generowanie tekstu. Możesz także generować obrazy na podstawie opisów tekstowych. Obrazy jako modalność są przydatne w MedTech, architekturze, turystyce, tworzeniu gier, marketingu i nie tylko. W tej lekcji pracujemy z dzisiejszymi modelami **GPT Image** na platformie Microsoft Foundry.

## Cele nauki

Pod koniec tej lekcji będziesz potrafił:

- Wyjaśnić, czym jest generowanie obrazów i gdzie jest przydatne.
- Zrozumieć rodzinę modeli `gpt-image` oraz jak różnią się od starszych modeli DALL·E.
- Zbudować aplikację do generowania obrazów oraz edytować obrazy.

## Czym jest generowanie obrazów?

Modele generujące obrazy tworzą obrazy na podstawie tekstowego polecenia. Nowoczesne modele takie jak `gpt-image` uczą się relacji między tekstem a obrazami podczas treningu, a następnie iteracyjnie przekształcają losowy szum w obraz odpowiadający twojemu poleceniu.

Dwie dobrze znane rodziny modeli obrazów to:

- **`gpt-image` (OpenAI)** — obecna generacja używana w tej lekcji. Obsługuje generowanie obrazów z tekstu oraz edycję obrazów (inpainting z maską).
- **Midjourney** — popularny model zewnętrzny z własną usługą i workflow opartym na Discordzie.

> Starsze modele obrazów OpenAI — **DALL·E 2** i **DALL·E 3** — są przestarzałe. DALL·E 3 nie jest już dostępne dla nowych wdrożeń, a funkcje takie jak `create_variation` istniały tylko w DALL·E 2. Do nowych aplikacji używaj modeli `gpt-image`.

Na platformie Microsoft Foundry, **`gpt-image-2`** jest najnowszym i najbardziej zaawansowanym modelem obrazów i jest domyślnie zalecany. `gpt-image-1.5` oraz `gpt-image-1-mini` są również ogólnie dostępne.

> **Ważne:** modele `gpt-image` zwracają wygenerowany obraz jako **base64** (`b64_json`), a nie jako URL. Twój kod dekoduje ciąg base64 do bajtów i zapisuje obraz — nie ma URL obrazu do pobrania.


## Tworzenie swojej pierwszej aplikacji do generowania obrazów

Czego potrzebujesz, aby zbudować aplikację do generowania obrazów? Potrzebujesz następujących bibliotek:

- **python-dotenv**, zaleca się korzystanie z tej biblioteki, aby przechowywać tajne dane w pliku *.env* z dala od kodu.
- **openai**, tej biblioteki użyjesz do interakcji z API OpenAI.
- **pillow**, do pracy z obrazami w Pythonie.

Jeśli jeszcze tego nie zrobiłeś, postępuj zgodnie z instrukcjami na stronie [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst), aby utworzyć zasób i model Microsoft Foundry. Wybierz **gpt-image-2** jako model (najnowszy model obrazów Azure OpenAI; DALL·E jest przestarzały).

1. Utwórz plik *.env* z następującą zawartością:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Znajdź te informacje w portalu Microsoft Foundry dla swojego zasobu w sekcji "Deployments".



1. Zbierz powyższe biblioteki do pliku o nazwie *requirements.txt*, w następujący sposób:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Następnie, utwórz wirtualne środowisko i zainstaluj biblioteki:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Dla Windows użyj następujących poleceń, aby stworzyć i aktywować swoje środowisko wirtualne:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodaj następujący kod w pliku o nazwie *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # skonfiguruj klienta Azure OpenAI (Microsoft Foundry).
    # Modele obrazów wymagają najnowszej wersji API — sprawdź w dokumentacji Foundry, której wersji wymaga Twój model.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Utwórz obraz za pomocą API generowania obrazów
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Wpisz tutaj swój tekst polecenia
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # np. "gpt-image-2"
        )
        # Ustaw katalog dla przechowywanego obrazu
        image_dir = os.path.join(os.curdir, 'images')

        # Jeśli katalog nie istnieje, utwórz go
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Zainicjuj ścieżkę obrazu (uwaga: typ pliku powinien być png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modele gpt-image zwracają obraz jako base64 (b64_json), a nie URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Wyświetl obraz w domyślnej przeglądarce obrazów
        image = Image.open(image_path)
        image.show()

    # przechwyć wyjątki
    except BadRequestError as err:
        print(err)

    ```

Wyjaśnijmy ten kod:

- Najpierw importujemy potrzebne biblioteki, w tym bibliotekę OpenAI, bibliotekę dotenv, moduł base64 oraz bibliotekę Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Następnie ładujemy zmienne środowiskowe z pliku *.env*.

    ```python
    # importuj dotenv
    dotenv.load_dotenv()
    ```

- Po tym konfigurujemy klienta Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Następnie generujemy obraz:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Wpisz tutaj tekst żądania
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Modele `gpt-image` zwracają obraz jako ciąg **base64** w `data[0].b64_json`. Dekodujemy go do bajtów i zapisujemy do pliku — nie ma URL do pobrania.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na koniec otwieramy obraz i używamy standardowej przeglądarki zdjęć do jego wyświetlenia:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Więcej informacji o generowaniu obrazu

Spójrzmy na parametry `images.generate`:

- **prompt** to tekstowy prompt używany do generowania obrazu. Tutaj jest to "Zając na koniu, trzymający lizaka, na mglistym łące, gdzie rosną narcyzy".
- **size** to rozmiar generowanego obrazu (na przykład `1024x1024`, `1536x1024`, `1024x1536` lub `"auto"`).
- **n** to liczba generowanych obrazów. Tutaj generujemy jeden.
- **model** to nazwa Twojego wdrożenia modelu obrazowego (na przykład `gpt-image-2`).

> Modele obrazowe nie przyjmują parametru `temperature` — to kontrola generowania tekstu. Aby uzyskać różnorodność, wywołaj API ponownie; aby zmniejszyć różnorodność, uczyn prompt bardziej szczegółowym.

## Dodatkowe możliwości generowania obrazu

Widziałeś, jak wygenerować obraz za pomocą kilku linijek Pythona. Modele `gpt-image` mogą również **edytować** istniejący obraz. Podając obraz, opcjonalną **maskę** (która zaznacza obszar do zmiany) oraz prompt, możesz zmienić część obrazu — na przykład dodać kapelusz naszemu zającowi.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edycje są również zwracane jako base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Obraz bazowy zawiera tylko zająca; obraz końcowy ma kapelusz na zającu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
